# Task 1 — Backfill Feature Pipeline (Olivais, Lisboa)

Run each cell **in order** with the ▶ button.

This notebook:
1. Installs `hopsworks` + dependencies
2. Takes your API keys (typed in securely, not saved)
3. Uploads your air-quality CSV
4. Loads historical **air quality** from the CSV
5. Downloads >1 year of historical **weather** from Open-Meteo
6. Registers **2 Feature Groups** in Hopsworks: `air_quality` and `weather`


## 1. Install dependencies

Takes ~2-3 minutes. Ignore any dependency-resolver warnings at the end.

In [ ]:
!pip install -q hopsworks great-expectations openmeteo-requests requests-cache retry-requests geopy xgboost
print("\n Install complete - continue to the next cell.")

## 2. Enter your API keys

You'll be prompted for each. Nothing is saved to disk or printed.

In [ ]:
import getpass, os

os.environ["HOPSWORKS_API_KEY"] = getpass.getpass("HOPSWORKS_API_KEY: ").strip()
os.environ["HOPSWORKS_PROJECT"] = input("HOPSWORKS_PROJECT (project name): ").strip()
os.environ["HOPSWORKS_HOST"]    = "app.hopsworks.ai"
AQICN_API_KEY = getpass.getpass("AQICN_API_KEY: ").strip()

print("\nKeys set.")

## 3. Upload your air quality CSV

Click **Choose Files** and pick `olivais-lisboa.csv` from:
`Downloads/ThesisWork/ScalableMLDL/Lab1/Lab1mlfs-book/data/`

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
csv_file = list(uploaded.keys())[0]
print(f"\nUploaded: {csv_file}")
pd.read_csv(csv_file, skipinitialspace=True).head()

## 4. Sensor configuration

Olivais, Lisboa — AQICN station uid **10513**.

In [ ]:
import datetime

country   = "Portugal"
city      = "Lisboa"
street     = "Olivais"
aqicn_url = "https://api.waqi.info/feed/@10513"
latitude  = 38.76888900025
longitude = -9.108055999671

today     = datetime.date.today()
yesterday = today - datetime.timedelta(days=1)

print(f"{street}, {city}, {country}  ({latitude}, {longitude})")

## 5. Connect to Hopsworks

A successful login prints your project URL.

In [ ]:
import hopsworks

project = hopsworks.login()
fs = project.get_feature_store()
print('')
print(f'Connected to project: {project.name}')


## 6. Store sensor location + API key as Hopsworks secrets

The **daily pipeline** (Task 2) reads these instead of hardcoding them.

In [ ]:
import json

secrets = hopsworks.get_secrets_api()

def put_secret(name, value):
    try:
        existing = secrets.get_secret(name)
        if existing is not None:
            existing.delete()
            print(f"Replaced existing {name}")
    except Exception:
        pass
    secrets.create_secret(name, value)
    print(f"Stored {name}")

put_secret("AQICN_API_KEY", AQICN_API_KEY)
put_secret("SENSOR_LOCATION_JSON", json.dumps({
    "country": country, "city": city, "street": street,
    "aqicn_url": aqicn_url, "latitude": latitude, "longitude": longitude,
}))

## 7. Load historical air quality from the CSV

Keep only `date` + `pm25`, drop rows with no reading, and tag with the sensor location.

In [ ]:
df = pd.read_csv(csv_file, parse_dates=['date'], skipinitialspace=True)
print(f"Raw rows: {len(df)}")

df_aq = df[['date', 'pm25']].copy()
df_aq['pm25'] = pd.to_numeric(df_aq['pm25'], errors='coerce').astype('float32')
df_aq = df_aq.dropna()

df_aq['country'] = country
df_aq['city']    = city
df_aq['street']  = street
df_aq['url']     = aqicn_url

print(f"Usable rows: {len(df_aq)}")
print(f"Date range : {df_aq['date'].min().date()} -> {df_aq['date'].max().date()}")
df_aq.head()

## 8. Download historical weather from Open-Meteo

Fetched from the earliest air-quality date up to yesterday, so the two
feature groups line up on `date` when we join them for training.

In [ ]:
import openmeteo_requests, requests_cache
from retry_requests import retry

start_date = df_aq['date'].min().strftime('%Y-%m-%d')
end_date   = str(yesterday)
print(f"Fetching weather: {start_date} -> {end_date}")

openmeteo = openmeteo_requests.Client(
    session=retry(requests_cache.CachedSession('.cache', expire_after=-1),
                  retries=5, backoff_factor=0.2))

resp = openmeteo.weather_api(
    "https://archive-api.open-meteo.com/v1/archive",
    params={"latitude": latitude, "longitude": longitude,
            "start_date": start_date, "end_date": end_date,
            "daily": ["temperature_2m_mean", "precipitation_sum",
                      "wind_speed_10m_max", "wind_direction_10m_dominant"]})[0]

daily = resp.Daily()
weather_df = pd.DataFrame({
    "date": pd.date_range(pd.to_datetime(daily.Time(), unit="s"),
                          pd.to_datetime(daily.TimeEnd(), unit="s"),
                          freq=pd.Timedelta(seconds=daily.Interval()),
                          inclusive="left"),
    "temperature_2m_mean":          daily.Variables(0).ValuesAsNumpy(),
    "precipitation_sum":            daily.Variables(1).ValuesAsNumpy(),
    "wind_speed_10m_max":           daily.Variables(2).ValuesAsNumpy(),
    "wind_direction_10m_dominant":  daily.Variables(3).ValuesAsNumpy(),
}).dropna()
weather_df['city'] = city

print(f"Weather rows: {len(weather_df)}")
weather_df.head()

## 9. Data validation (Great Expectations)

Hopsworks enforces these on every insert, so bad data is caught at write
time rather than silently poisoning the model.

In [ ]:
import great_expectations as ge

aq_suite = ge.core.ExpectationSuite(expectation_suite_name="aq_expectation_suite")
aq_suite.add_expectation(ge.core.ExpectationConfiguration(
    expectation_type="expect_column_min_to_be_between",
    kwargs={"column": "pm25", "min_value": -0.1, "max_value": 500.0, "strict_min": True}))

weather_suite = ge.core.ExpectationSuite(expectation_suite_name="weather_expectation_suite")
for col in ["precipitation_sum", "wind_speed_10m_max"]:
    weather_suite.add_expectation(ge.core.ExpectationConfiguration(
        expectation_type="expect_column_min_to_be_between",
        kwargs={"column": col, "min_value": -0.1, "max_value": 1000.0, "strict_min": True}))

print("Expectation suites ready.")

## 10. Feature Group 1 of 2 — `air_quality`

`primary_key` + `event_time` identify each row. Insertion runs in the
background; the job link it prints is worth keeping for the defence.

In [ ]:
air_quality_fg = fs.get_or_create_feature_group(
    name='air_quality',
    description='Air Quality characteristics of each day',
    version=1,
    primary_key=['country', 'city', 'street'],
    event_time="date",
    expectation_suite=aq_suite,
)
air_quality_fg.insert(df_aq, wait=True)
print("\nair_quality feature group written.")

In [ ]:
for col, desc in {
    "date":    "Date of measurement of air quality",
    "country": "Country where the air quality was measured",
    "city":    "City where the air quality was measured",
    "street":  "Street in the city where the air quality was measured",
    "pm25":    "Particles less than 2.5 micrometers in diameter (fine particles) pose health risk",
    "url":     "URL of the AQICN sensor feed",
}.items():
    air_quality_fg.update_feature_description(col, desc)
print("Descriptions added.")

## 11. Feature Group 2 of 2 — `weather`

In [ ]:
weather_fg = fs.get_or_create_feature_group(
    name='weather',
    description='Weather characteristics of each day',
    version=1,
    primary_key=['city'],
    event_time="date",
    expectation_suite=weather_suite,
)
weather_fg.insert(weather_df, wait=True)
print("\nweather feature group written.")

In [ ]:
for col, desc in {
    "date":                        "Date of measurement of weather",
    "city":                        "City where the weather was measured",
    "temperature_2m_mean":         "Mean temperature at 2m above ground (Celsius)",
    "precipitation_sum":           "Total daily precipitation (mm)",
    "wind_speed_10m_max":          "Maximum wind speed at 10m above ground (km/h)",
    "wind_direction_10m_dominant": "Dominant wind direction at 10m above ground (degrees)",
}.items():
    weather_fg.update_feature_description(col, desc)
print("Descriptions added.")

## 12. Verify — Task 1 complete

Both feature groups should be listed with non-zero row counts.

In [ ]:
print("=" * 55)
print("TASK 1 COMPLETE - 2 Feature Groups registered")
print("=" * 55)
for name in ("air_quality", "weather"):
    fg = fs.get_feature_group(name=name, version=1)
    print(f"\n{name} (v{fg.version})")
    print(f"  rows     : {len(fg.read()):,}")
    print(f"  features : {[f.name for f in fg.features]}")

print(f"\nView them at: {project.get_url()}")